# Download del dataset

In [ ]:
!pip install SoccerNet
from SoccerNet.Downloader import SoccerNetDownloader
mySoccerNetDownloader = SoccerNetDownloader(LocalDirectory="./dataset")
mySoccerNetDownloader.downloadDataTask(task="tracking", split=["train","test"])

# Preparazione del dataset in formato YOLO

In [ ]:
import os
import pandas as pd
from glob import glob
import re
import yaml
import shutil
from tqdm import tqdm

# Modificare i percorsi secondo necessità
# Percorso alla cartella SoccerNet scaricata
BASE_PATH = "./dataset" 

# Cartella di lavoro per il dataset YOLO
DATA_DIR = "./working/data"

# Definizione delle Classi del Modello YOLO
labels = ['person']

# Mappatura Stringa Dataset -> ID Numerico Modello YOLO
# Nota: 'ball' è esclusa 
label_dict = {
    'goalkeeper': 0,   
    'goalkeepers': 0,  
    'player': 0,       
    'referee': 0       
}

# Creazione delle cartelle necessarie
yolo_train_img_dir = os.path.join(DATA_DIR, "images/train")
yolo_val_img_dir = os.path.join(DATA_DIR, "images/val")
yolo_train_label_dir = os.path.join(DATA_DIR, "labels/train")
yolo_val_label_dir = os.path.join(DATA_DIR, "labels/val")

dirs_to_make = [yolo_train_img_dir, yolo_val_img_dir, yolo_train_label_dir, yolo_val_label_dir]
for d in dirs_to_make:
    os.makedirs(d, exist_ok=True)

# Percorso al file YAML di configurazione
yaml_file = os.path.join(DATA_DIR, 'data.yaml')

all_train_dirs = sorted(glob(os.path.join(BASE_PATH, "train/SNMOT*")))
all_test_dirs = sorted(glob(os.path.join(BASE_PATH, "test/SNMOT*")))


split_dirs = {
    'train': all_train_dirs,
    'validate': all_test_dirs
}

print(f"Video di Training: {len(all_train_dirs)}")
print(f"Video di Validazione: {len(all_test_dirs)}")

def get_yolo_format_bbox(img_w, img_h, box):
    """
    Converte bbox da (top-left-x, top-left-y, w, h)
    a formato YOLO (center-x, center-y, w, h) normalizzato.
    """
    x, y, w, h = box
    xc = x + (w / 2)
    yc = y + (h / 2)

    # Normalizzazione tra 0 e 1
    return [xc/img_w, yc/img_h, w/img_w, h/img_h]

def get_info(info_path):
    """
    Legge il file gameinfo.ini per mappare trackletID -> classe stringa.
    Filtra automaticamente le classi non presenti in label_dict.
    """
    results = []
    if not os.path.exists(info_path):
        return {}

    with open(info_path, 'r') as f:
        for line in f:
            # Regex per estrarre ID e Nome Classe
            m = re.match(r'trackletID_(\d+)=\s*(\S*).*', line.replace(';', ' '))
            if m:
                track_id, label_name = m.group(1), m.group(2)
                # Controlliamo se la label è una di quelle che ci interessano (incluse nel dict)
                # Nota: qui accettiamo tutto, il filtro vero avviene dopo col map()
                results.append([track_id, label_name])

    if not results:
        return {}

    return pd.DataFrame(results, columns=['id', 'lbl']).set_index('id').to_dict()['lbl']

for split, directories in split_dirs.items():
    if split == 'train':
        curr_img_dir = yolo_train_img_dir
        curr_label_dir = yolo_train_label_dir
    else:
        curr_img_dir = yolo_val_img_dir
        curr_label_dir = yolo_val_label_dir

    print(f"Elaborazione split: {split}...")

    for this_dir in tqdm(directories):
        video_name = os.path.basename(this_dir)
        info_file = os.path.join(this_dir, 'gameinfo.ini')
        det_file = os.path.join(this_dir, 'gt', 'gt.txt')

        # Carica Mappatura ID -> Classe
        info_dict = get_info(info_file)

        if not os.path.exists(det_file):
            print(f"Warning: GT file mancante per {video_name}")
            continue

        # Annotazioni
        # Colonne SoccerNet file GT: frame, player_id, x, y, w, h
        det_df = pd.read_csv(det_file,
                             names=['frame', 'player', 'x', 'y', 'w', 'h'],
                             usecols=[0, 1, 2, 3, 4, 5])

        # Mappa l'ID numerico del giocatore alla stringa della classe
        det_df['label_str'] = det_df.player.astype(str).map(info_dict)

        # Mappa la stringa della classe all'ID numerico YOLO CORRETTO
        det_df['label_id'] = det_df['label_str'].map(label_dict)

        # Rimuovi tutte le righe che non hanno una label valida
        det_df = det_df.dropna(subset=['label_id'])
        det_df['label_id'] = det_df['label_id'].astype(int)

        img_width, img_height = 1920, 1080

        for frame_idx, grp_df in det_df.groupby('frame'):
            frame_str = f'{frame_idx:06}'
            src_img_path = os.path.join(this_dir, 'img1', f'{frame_str}.jpg')

            dst_img_name = f'{video_name}_{frame_str}.jpg'
            dst_img_path = os.path.join(curr_img_dir, dst_img_name)
            dst_label_path = os.path.join(curr_label_dir, f'{video_name}_{frame_str}.txt')

            if os.path.exists(src_img_path):
                shutil.copy(src_img_path, dst_img_path)
            else:
                continue

            with open(dst_label_path, 'w') as f:
                for row in grp_df.itertuples():
                    # Conversione bbox
                    bbox_norm = get_yolo_format_bbox(img_width, img_height, [row.x, row.y, row.w, row.h])

                    # Formato riga YOLO: class_id center_x center_y w h
                    line = f"{row.label_id} {bbox_norm[0]:.6f} {bbox_norm[1]:.6f} {bbox_norm[2]:.6f} {bbox_norm[3]:.6f}\n"
                    f.write(line)

# Configurazione YAML corretta
data_yaml = dict(
    path = DATA_DIR,
    train = "images/train",
    val = "images/val",
    nc = len(labels),             
    names = labels     
)

with open(yaml_file, 'w') as outfile:
    yaml.dump(data_yaml, outfile, default_flow_style=False)

print("Conversione completata! Dataset CORRETTO pronto.")

# Campionamento del dataset originale

In [ ]:
import os
import shutil
from glob import glob
from tqdm import tqdm

# Modificare i percorsi secondo necessità
# Percorso dove si trova il dataset completo 
SOURCE_DIR = "./working/data"
# Percorso dove verrà creato il nuovo dataset campionato
DEST_DIR = "./working/data_sampled"
# Ogni quanti frame campionare
SAMPLE_RATE = 10

def create_sampled_dataset():
    # Struttura delle cartelle
    splits = ['train', 'val']
    subdirs = ['images', 'labels']

    for split in splits:
        for subdir in subdirs:
            os.makedirs(os.path.join(DEST_DIR, subdir, split), exist_ok=True)


    yaml_file = os.path.join(DEST_DIR, "data.yaml")
    # Configurazione YAML corretta
    sampled_yaml = dict(
        path = DEST_DIR,
        train = "images/train",
        val = "images/val",
        nc = len(labels),             
        names = labels     
    )

    with open(yaml_file, 'w') as outfile:
        yaml.dump(sampled_yaml, outfile, default_flow_style=False)

    print(f"🔄 Inizio campionamento (1 frame ogni {SAMPLE_RATE})...")

    total_copied = 0

    for split in splits:
        img_source_dir = os.path.join(SOURCE_DIR, "images", split)
        label_source_dir = os.path.join(SOURCE_DIR, "labels", split)

        img_dest_dir = os.path.join(DEST_DIR, "images", split)
        label_dest_dir = os.path.join(DEST_DIR, "labels", split)

        images = glob(os.path.join(img_source_dir, "*.jpg"))

        print(f"Elaborazione split '{split}': {len(images)} immagini totali.")

        for img_path in tqdm(images):
            filename = os.path.basename(img_path)

            # Il formato è: NomeVideo_FrameNumber.jpg (es. SNMOT-060_000125.jpg)
            # Estraiamo la parte numerica finale
            try:
                name_without_ext = os.path.splitext(filename)[0]
                frame_str = name_without_ext.split('_')[-1] # Prende l'ultima parte dopo l'underscore
                frame_idx = int(frame_str)

                # LOGICA DI CAMPIONAMENTO
                # Teniamo il frame se è divisibile per SAMPLE_RATE 
                if frame_idx % SAMPLE_RATE == 0:

                    # Copia Immagine
                    shutil.copy(img_path, os.path.join(img_dest_dir, filename))

                    # Copia Label corrispondente
                    label_filename = name_without_ext + ".txt"
                    src_label = os.path.join(label_source_dir, label_filename)

                    if os.path.exists(src_label):
                        shutil.copy(src_label, os.path.join(label_dest_dir, label_filename))

                    total_copied += 1

            except ValueError:
                print(f"Skipping file con nome non standard: {filename}")
                continue

    print(f"Campionamento completato!")
    print(f"Nuovo dataset salvato in: {DEST_DIR}")
    print(f"Totale immagini nel nuovo dataset: {total_copied}")

create_sampled_dataset()

# Creazione dataset per l'addestramento di modelli per il Re-ID

In [ ]:
import os
import cv2
import configparser
import numpy as np
from tqdm import tqdm

# Modificare i percorsi secondo necessità
SOURCE_ROOT_DIR = "./dataset" 
OUTPUT_DIR = "./reid_dataset" 
IMG_SIZE = (256, 128)

# Minima altezza in pixel per considerare una persona
MIN_HEIGHT = 40       
# Numero di frame da saltare tra un campione e l'altro
FRAME_SAMPLE_RATE = 10 

def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

# Estrae gli ID validi delle persone dal file gameinfo.ini
def parse_gameinfo(ini_path):
    valid_ids = set()
    config = configparser.ConfigParser()
    try:
        config.read(ini_path)
        if 'Sequence' in config:
            section = config['Sequence']
            for key in section:
                if key.startswith('trackletid_'):
                    try:
                        tid = int(key.split('_')[1])
                    except ValueError: continue
                    value = section[key].lower()
                    if "ball" not in value:
                        valid_ids.add(tid)
    except Exception: return set()
    return valid_ids

# Processa una singola sequenza
def process_sequence(seq_path, output_split_folder):
    seq_name = os.path.basename(seq_path)
    gt_path = os.path.join(seq_path, "gt", "gt.txt")
    ini_path = os.path.join(seq_path, "gameinfo.ini")
    img_dir = os.path.join(seq_path, "img1")
    
    if not os.path.exists(gt_path): return 0 

    valid_person_ids = parse_gameinfo(ini_path)
    if not valid_person_ids: return 0

    try:
        gt_data = np.loadtxt(gt_path, delimiter=',')
    except: return 0

    if not os.path.exists(img_dir): return 0
    available_imgs = set(os.listdir(img_dir))

    count_crops = 0
    frames_gt = {}
    
    for row in gt_data:
        frame_id = int(row[0])
        
        if frame_id % FRAME_SAMPLE_RATE != 0:
            continue
        
        if frame_id not in frames_gt: frames_gt[frame_id] = []
        frames_gt[frame_id].append(row)

    # Itera SOLO sui frame campionati
    for frame_id, detections in frames_gt.items():
        img_name = f"{frame_id:06d}.jpg"
        if img_name not in available_imgs: continue
            
        img_path = os.path.join(img_dir, img_name)
        frame = cv2.imread(img_path)
        if frame is None: continue
        
        h_img, w_img = frame.shape[:2]

        for row in detections:
            tid = int(row[1])
            if tid not in valid_person_ids: continue
            
            x1, y1, w, h = int(row[2]), int(row[3]), int(row[4]), int(row[5])
            if h < MIN_HEIGHT: continue
            
            x2, y2 = min(w_img, x1+w), min(h_img, y1+h)
            x1, y1 = max(0, x1), max(0, y1)
            
            crop = frame[y1:y2, x1:x2]
            if crop.size == 0: continue
            
            crop = cv2.resize(crop, (IMG_SIZE[1], IMG_SIZE[0]))
            
            unique_id = f"{seq_name}_{tid}"
            save_folder = os.path.join(output_split_folder, unique_id)
            ensure_dir(save_folder)
            
            cv2.imwrite(os.path.join(save_folder, img_name), crop)
            count_crops += 1
            
    return count_crops

if __name__ == "__main__":
    SPLITS_TO_PROCESS = ["train", "test"] 
    
    print(f"Generazione Dataset (1 frame ogni {FRAME_SAMPLE_RATE})...")
    
    for split in SPLITS_TO_PROCESS:
        source_split_path = os.path.join(SOURCE_ROOT_DIR, split)
        output_split_path = os.path.join(OUTPUT_DIR, split)
        
        if not os.path.exists(source_split_path): continue
            
        ensure_dir(output_split_path)
        sequences = sorted([f.path for f in os.scandir(source_split_path) if f.is_dir()])
        
        total_crops = 0
        for seq in tqdm(sequences, desc=f"Processing {split}"):
            crops = process_sequence(seq, output_split_path)
            total_crops += crops
            
        print(f"Finito {split}: {total_crops} crops.")